In [22]:
import sys
import torch

print("Python Version:", sys.version)
print("PyTorch Version:", torch.__version__)
print("CUDA Version (PyTorch):", torch.version.cuda)
print("CUDA Available:", torch.cuda.is_available())

Python Version: 3.13.5 (tags/v3.13.5:6cb20a2, Jun 11 2025, 16:15:46) [MSC v.1943 64 bit (AMD64)]
PyTorch Version: 2.7.1+cu118
CUDA Version (PyTorch): 11.8
CUDA Available: True


In [23]:
# Cell 0: imports, configuration, SystemVerilog Tree-sitter parser

from __future__ import annotations

import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import fitz  # PyMuPDF

# Tree-sitter core API
from tree_sitter import Language, Parser, Node  # Parser(language), Node.children etc.

# SystemVerilog grammar: install e.g. `pip install tree-sitter-verilog`
# and adjust the import if you use a different package.
import tree_sitter_verilog as tsv  # this package exposes `language()` pointer


# --- Project root and data/work layout ---

ROOT = Path.cwd().parent  # same as in 01
DATA_DIR = ROOT / "data"
WORK_DIR = ROOT / "work"

# UVM User Guide 1.2 (manual)
MANUAL_PDF_PATH = DATA_DIR / "uvm_users_guide_1.2.pdf"
MANUAL_WORK_DIR = WORK_DIR / "work_manual"
MANUAL_CONTENT_LIST = (
    MANUAL_WORK_DIR
    / "mineru_out"
    / "uvm_users_guide_1.2"
    / "auto"
    / "uvm_users_guide_1.2_content_list.json"
)
MANUAL_OUT_JSONL = MANUAL_WORK_DIR / "json_out" / "uvm_blocks_augmented.jsonl"
MANUAL_URI = "/pdf/uvm_users_guide_1.2.pdf"
STD_TAG = "UVM-1.2"

# If you already have a different layout for the class reference,
# just correct these three constants and the second call at the bottom.
CLASS_PDF_PATH = DATA_DIR / "UVM_Class_Reference_Manual_1.2.pdf"
CLASS_WORK_DIR = WORK_DIR / "work_class_reference"  # TODO: set to your actual folder
CLASS_CONTENT_LIST = (
    CLASS_WORK_DIR
    / "mineru_out"
    / "UVM_Class_Reference_Manual_1.2"
    / "auto"
    / "UVM_Class_Reference_Manual_1.2_content_list.json"
)
CLASS_OUT_JSONL = (
    CLASS_WORK_DIR / "json_out" / "uvm_class_reference_blocks_augmented.jsonl"
)
CLASS_URI = "/pdf/UVM_Class_Reference_Manual_1.2.pdf"


# --- SystemVerilog Tree-sitter parser (0.25.x API) ---

# The grammar wheel (tree_sitter_verilog) exposes a `language()` function that
# returns the C pointer; py-tree-sitter wraps it via Language(ptr). :contentReference[oaicite:1]{index=1}
SV_LANGUAGE = Language(tsv.language())
SV_PARSER = Parser(SV_LANGUAGE)  # Parser(language=...)

print("ROOT:", ROOT)
print("Manual content_list:", MANUAL_CONTENT_LIST)
print("Manual out JSONL:", MANUAL_OUT_JSONL)
print("Tree-sitter language name:", SV_LANGUAGE.name)


ROOT: c:\Users\41v1r\NEU\NLP\UVM-RAG
Manual content_list: c:\Users\41v1r\NEU\NLP\UVM-RAG\work\work_manual\mineru_out\uvm_users_guide_1.2\auto\uvm_users_guide_1.2_content_list.json
Manual out JSONL: c:\Users\41v1r\NEU\NLP\UVM-RAG\work\work_manual\json_out\uvm_blocks_augmented.jsonl
Tree-sitter language name: None


In [24]:
@dataclass
class TocNode:
    level: int          # outline level (1=chapter, etc.)
    id: Optional[str]   # numeric id, e.g. "4.4.1" if present
    title: str          # title text without numeric prefix
    start: int          # 1-based start page
    end: int            # 1-based inclusive end page


_NUMERIC_PREFIX_RE = re.compile(r"^\s*(\d+(?:\.\d+)*)\s+(.+)$")


def _normalize_text(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()


def _split_numeric_prefix(raw_title: str) -> Tuple[Optional[str], str]:
    """
    '4.4.1 Verification ...' -> ('4.4.1', 'Verification ...')
    'Introduction' -> (None, 'Introduction')
    """
    m = _NUMERIC_PREFIX_RE.match(raw_title)
    if not m:
        return None, _normalize_text(raw_title)
    sec_id = m.group(1)
    sec_title = _normalize_text(m.group(2))
    return sec_id, sec_title


def build_toc_intervals(pdf_path: Path) -> List[TocNode]:
    """
    Use PyMuPDF outline to build TOC intervals [start, end] for each entry.
    """
    doc = fitz.open(pdf_path.as_posix())
    raw_toc = doc.get_toc(simple=True)  # [[level, title, page], ...]
    nodes: List[TocNode] = []

    for level, raw_title, page in raw_toc:
        sec_id, sec_title = _split_numeric_prefix(raw_title)
        nodes.append(
            TocNode(
                level=int(level),
                id=sec_id,
                title=sec_title,
                start=int(page),
                end=int(page),  # provisional; updated later
            )
        )

    page_count = doc.page_count
    doc.close()

    for i, node in enumerate(nodes):
        if i + 1 < len(nodes):
            next_start = nodes[i + 1].start
            node.end = max(node.start, next_start - 1)
        else:
            node.end = page_count

    print(f"[TOC] {pdf_path.name}: {len(nodes)} entries")
    return nodes


def page_to_section_meta(page: int, toc_nodes: List[TocNode]) -> Dict[str, Any]:
    """
    Map page -> section metadata by selecting the last TOC entry
    whose [start, end] contains this page.
    """
    chosen: Optional[TocNode] = None
    for node in toc_nodes:
        if node.start <= page <= node.end:
            chosen = node

    if chosen is None:
        return {
            "section_title": None,
            "header_path": [],
            "chapter": None,
            "section": None,
            "subsection": None,
        }

    meta: Dict[str, Any] = {
        "section_title": chosen.title,
        "header_path": [],
        "chapter": None,
        "section": None,
        "subsection": None,
    }

    if chosen.id:
        parts = chosen.id.split(".")
        meta["header_path"] = parts
        if len(parts) >= 1:
            meta["chapter"] = parts[0]
        if len(parts) >= 2:
            meta["section"] = ".".join(parts[:2])
        if len(parts) >= 3:
            meta["subsection"] = ".".join(parts[:3])

    return meta


In [25]:
def compute_header_footer_bands(blocks: List[Dict[str, Any]]) -> Dict[int, Tuple[float, float]]:
    """
    For each page_idx (0-based) compute (header_y_max, footer_y_min).

    We consider roughly top 12% as header and bottom 12% as footer based on
    min/max y across blocks on that page.
    """
    per_page: Dict[int, List[Tuple[float, float]]] = {}
    for b in blocks:
        if "page_idx" not in b or "bbox" not in b:
            continue
        page_idx = int(b["page_idx"])
        x0, y0, x1, y1 = b["bbox"]
        per_page.setdefault(page_idx, []).append((float(y0), float(y1)))

    bands: Dict[int, Tuple[float, float]] = {}
    for page_idx, ys in per_page.items():
        y0s = [p[0] for p in ys]
        y1s = [p[1] for p in ys]
        ymin, ymax = min(y0s), max(y1s)
        h = max(1.0, ymax - ymin)
        header_y_max = ymin + 0.12 * h
        footer_y_min = ymax - 0.12 * h
        bands[page_idx] = (header_y_max, footer_y_min)

    return bands


def is_header_or_footer_block(
    block: Dict[str, Any],
    bands: Dict[int, Tuple[float, float]],
) -> bool:
    """
    Treat a block as header/footer if its bbox lies entirely in top or bottom band.
    """
    if "page_idx" not in block or "bbox" not in block:
        return False

    page_idx = int(block["page_idx"])
    if page_idx not in bands:
        return False

    header_y_max, footer_y_min = bands[page_idx]
    _, y0, _, y1 = block["bbox"]
    y0 = float(y0)
    y1 = float(y1)

    if y1 <= header_y_max:
        return True
    if y0 >= footer_y_min:
        return True
    return False


In [26]:
# We use Tree-sitter itself to decide whether text is SystemVerilog or not.
# A snippet is 'OK' SV code iff:
#   - parse contains NO ERROR nodes, and
#   - there is at least one 'symbol' node (class/module/task/function/package/...).


SV_SYMBOL_NODE_TYPES = {
    "class_declaration",
    "function_declaration",
    "task_declaration",
    "module_declaration",
    "interface_declaration",
    "covergroup_declaration",
    "program_declaration",
    "package_declaration",
}


def count_error_nodes(root) -> int:
    cnt = 0
    stack = [root]
    while stack:
        node = stack.pop()
        if node.type == "ERROR":
            cnt += 1
        stack.extend(node.children)
    return cnt


def has_sv_symbol_node(root) -> bool:
    stack = [root]
    while stack:
        node = stack.pop()
        if node.type in SV_SYMBOL_NODE_TYPES:
            return True
        stack.extend(node.children)
    return False


def parse_sv_stats(text: str, parser: Parser) -> Tuple[bool, int, Any]:
    """
    Parse 'text' as SystemVerilog and return:
      - has_symbol: whether AST contains at least one SV symbol node
                    (class/function/task/module/typedef/...),
      - error_count: number of ERROR nodes,
      - root: the AST root node.

    We will NOT decide code vs text here; that will be done incrementally
    in process_document using thresholds on error_count.
    """
    if not text.strip():
        return False, 0, None

    src = text.encode("utf-8")
    tree = parser.parse(src)
    root = tree.root_node

    has_symbol = has_sv_symbol_node(root)
    err_cnt = count_error_nodes(root)
    return has_symbol, err_cnt, root




def ts_node_to_meta(node, source_bytes: bytes, max_depth: int = 2) -> Dict[str, Any]:
    """
    Convert a Tree-sitter node into a JSON-serializable dict for debugging / inspection.
    """
    meta: Dict[str, Any] = {
        "type": node.type,
        "is_named": node.is_named,
        "start_byte": node.start_byte,
        "end_byte": node.end_byte,
        "start_point": [node.start_point[0], node.start_point[1]],
        "end_point": [node.end_point[0], node.end_point[1]],
    }
    if max_depth <= 0:
        meta["children"] = []
        return meta

    children_meta: List[Dict[str, Any]] = []
    for child in node.children:
        if not child.is_named:
            continue
        children_meta.append(ts_node_to_meta(child, source_bytes, max_depth=max_depth - 1))
    meta["children"] = children_meta
    return meta


In [27]:
def process_document(
    *,
    pdf_path: Path,
    content_list_path: Path,
    std: str,
    uri: str,
    out_jsonl_path: Path,
    parser: Parser,
) -> None:
    """
    Process one document (UVM Users Guide or Class Reference):

    - Load MinerU content_list.json.
    - Sort blocks by (page_idx, y0, x0).
    - Build TOC intervals and header/footer bands.
    - For each block:
        * Skip header/footer blocks.
        * TABLE: emit one 'table' record (caption, body, footnote, img_path).
        * IMAGE: emit one 'image' record (caption, footnote, img_path).
        * TEXT: incremental SV detection using Tree-sitter only:
            - maintain a code buffer on each page;
            - for each new line:
                · candidate = buffer + line;
                · if parse_sv_ok(candidate): buffer = candidate;
                · else:
                   · flush buffer as 'sv_code' record (if non-empty);
                   · emit this line as normal 'text'.
    - SV snippets do not cross pages; we flush on page change.
    - Write a single JSONL mixing 'text' | 'table' | 'image' | 'sv_code'.
    """

    # --- Load MinerU blocks ---
    with content_list_path.open("r", encoding="utf-8") as f:
        blocks: List[Dict[str, Any]] = json.load(f)

    # Sorting is robust to missing bbox
    def _block_sort_key(b: Dict[str, Any]) -> Tuple[int, float, float]:
        page_idx = int(b.get("page_idx", 0))
        bbox = b.get("bbox") or [0.0, 0.0, 0.0, 0.0]
        return (page_idx, float(bbox[1]), float(bbox[0]))

    blocks.sort(key=_block_sort_key)

    # --- TOC & header/footer ---
    toc_nodes = build_toc_intervals(pdf_path)
    bands = compute_header_footer_bands(blocks)

    out_jsonl_path.parent.mkdir(parents=True, exist_ok=True)
    records: List[Dict[str, Any]] = []

    # --- SV snippet state (per page) ---
    current_page_idx: Optional[int] = None
    sv_buffer_lines: List[str] = []
    sv_page_from: Optional[int] = None
    sv_page_to: Optional[int] = None
    sv_meta: Optional[Dict[str, Any]] = None
    sv_error_count: int = 0


    def flush_sv_buffer() -> None:
        """
        Flush current code buffer as 'sv_code' record.
        Assumes the current buffer had been accepted incrementally.
        """
        nonlocal sv_buffer_lines, sv_page_from, sv_page_to, sv_meta, sv_error_count

        if not sv_buffer_lines or sv_meta is None:
            sv_buffer_lines = []
            sv_page_from = None
            sv_page_to = None
            sv_meta = None
            sv_error_count = 0
            return

        code_text = "\n".join(sv_buffer_lines).strip()
        if not code_text:
            sv_buffer_lines = []
            sv_page_from = None
            sv_page_to = None
            sv_meta = None
            sv_error_count = 0
            return

        src = code_text.encode("utf-8")
        tree = parser.parse(src)
        root = tree.root_node
        ts_root = ts_node_to_meta(root, src, max_depth=2)

        rec: Dict[str, Any] = {
            "type": "sv_code",
            "std": std,
            "uri": uri,
            "page_from": sv_page_from,
            "page_to": sv_page_to,
            "anchor": f"#page={sv_page_from}",
            "content": code_text,
            "ts_root": ts_root,
        }
        rec.update(sv_meta)
        records.append(rec)

        sv_buffer_lines = []
        sv_page_from = None
        sv_page_to = None
        sv_meta = None
        sv_error_count = 0

    # --- Main scan over blocks ---
    for block in blocks:
        if "page_idx" not in block:
            continue

        page_idx = int(block["page_idx"])
        page = page_idx + 1

        # New page: SV snippets cannot cross pages
        if current_page_idx is None or page_idx != current_page_idx:
            flush_sv_buffer()
            current_page_idx = page_idx

        # Skip header/footer blocks for all types
        if is_header_or_footer_block(block, bands):
            continue

        section_meta = page_to_section_meta(page, toc_nodes)

        # Robust type detection
        btype = block.get("type")
        if btype is None:
            if "table_body" in block or "table_caption" in block:
                btype = "table"
            elif "image_caption" in block or "img_path" in block:
                btype = "image"
            elif "text" in block or "content" in block:
                btype = "text"
            else:
                # Utility/meta block; ignore
                continue

        # ---------------- TABLE ----------------
        if btype == "table":
            flush_sv_buffer()
            table_caption = " ".join(block.get("table_caption", [])).strip()
            table_footnote = " ".join(block.get("table_footnote", [])).strip()

            rec = {
                "type": "table",
                "std": std,
                "uri": uri,
                "page_from": page,
                "page_to": page,
                "anchor": f"#page={page}",
                "table_caption": table_caption,
                "table_body_html": block.get("table_body", ""),
                "table_footnote": table_footnote,
                "img_path": block.get("img_path"),
            }
            rec.update(section_meta)
            records.append(rec)
            continue

        # ---------------- IMAGE ----------------
        if btype == "image":
            flush_sv_buffer()
            image_caption = " ".join(block.get("image_caption", [])).strip()
            image_footnote = " ".join(block.get("image_footnote", [])).strip()

            rec = {
                "type": "image",
                "std": std,
                "uri": uri,
                "page_from": page,
                "page_to": page,
                "anchor": f"#page={page}",
                "image_caption": image_caption,
                "image_footnote": image_footnote,
                "img_path": block.get("img_path"),
            }
            rec.update(section_meta)
            records.append(rec)
            continue
        
        
                # ---------------- TEXT ----------------
        if btype == "text":
            raw_text = (block.get("text") or block.get("content") or "").strip()
            if not raw_text:
                continue

            lines = [ln for ln in raw_text.splitlines() if ln.strip()]
            if not lines:
                continue

            # Thresholds: tune if needed
            MAX_SV_LINES = 80            # do not let one snippet grow unbounded
            MAX_ERR_ABS = 3            # absolute upper bound on error nodes
            MAX_ERR_DELTA = 2           # allowed increase in errors when adding a line

            for ln in lines:
                ln_clean = ln.strip()

                # If current snippet already too long, flush it first
                if sv_buffer_lines and len(sv_buffer_lines) >= MAX_SV_LINES:
                    flush_sv_buffer()

                # Build candidate text by appending this line
                candidate_lines = (
                    sv_buffer_lines + [ln_clean] if sv_buffer_lines else [ln_clean]
                )
                candidate_text = "\n".join(candidate_lines)

                has_symbol, err_cnt, _root = parse_sv_stats(candidate_text, parser)

                if not sv_buffer_lines:
                    # No open SV snippet yet
                    if has_symbol and err_cnt <= MAX_ERR_ABS:
                        # Start a new SV snippet
                        sv_buffer_lines = candidate_lines
                        sv_error_count = err_cnt
                        sv_page_from = page
                        sv_page_to = page
                        sv_meta = {
                            "section_title": section_meta.get("section_title"),
                            "header_path": section_meta.get("header_path", []),
                            "chapter": section_meta.get("chapter"),
                            "section": section_meta.get("section"),
                            "subsection": section_meta.get("subsection"),
                        }
                    else:
                        # Treat as plain text
                        rec = {
                            "type": "text",
                            "std": std,
                            "uri": uri,
                            "page_from": page,
                            "page_to": page,
                            "anchor": f"#page={page}",
                            "content": ln_clean,
                        }
                        rec.update(section_meta)
                        records.append(rec)
                else:
                    # We already have an SV snippet buffer
                    err_delta = err_cnt - sv_error_count

                    # If still symbol and errors do not blow up, keep accumulating
                    if has_symbol and err_cnt <= MAX_ERR_ABS and err_delta <= MAX_ERR_DELTA:
                        sv_buffer_lines = candidate_lines
                        sv_error_count = err_cnt
                        sv_page_to = page
                    else:
                        # Adding this line ruins the parse too much: flush old snippet,
                        # then treat this line fresh (either new snippet or text).
                        flush_sv_buffer()

                        # Try this line alone as a fresh candidate
                        has_symbol1, err_cnt1, _root1 = parse_sv_stats(ln_clean, parser)
                        if has_symbol1 and err_cnt1 <= MAX_ERR_ABS:
                            sv_buffer_lines = [ln_clean]
                            sv_error_count = err_cnt1
                            sv_page_from = page
                            sv_page_to = page
                            sv_meta = {
                                "section_title": section_meta.get("section_title"),
                                "header_path": section_meta.get("header_path", []),
                                "chapter": section_meta.get("chapter"),
                                "section": section_meta.get("section"),
                                "subsection": section_meta.get("subsection"),
                            }
                        else:
                            rec = {
                                "type": "text",
                                "std": std,
                                "uri": uri,
                                "page_from": page,
                                "page_to": page,
                                "anchor": f"#page={page}",
                                "content": ln_clean,
                            }
                            rec.update(section_meta)
                            records.append(rec)

            continue  # finished TEXT block


        # Other types: flush SV and skip
        flush_sv_buffer()
        continue

    # End of loop: flush last SV block if any
    flush_sv_buffer()

    # --- Write JSONL ---
    with out_jsonl_path.open("w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    print(f"[01_5] {pdf_path.name}: wrote {len(records)} records to {out_jsonl_path}")


In [28]:
# UVM Users Guide 1.2
process_document(
    pdf_path=MANUAL_PDF_PATH,
    content_list_path=MANUAL_CONTENT_LIST,
    std=STD_TAG,
    uri=MANUAL_URI,
    out_jsonl_path=MANUAL_OUT_JSONL,
    parser=SV_PARSER,
)

# UVM Class Reference 1.2
process_document(
    pdf_path=CLASS_PDF_PATH,
    content_list_path=CLASS_CONTENT_LIST,
    std=STD_TAG,
    uri=CLASS_URI,
    out_jsonl_path=CLASS_OUT_JSONL,
    parser=SV_PARSER,
)


[TOC] uvm_users_guide_1.2.pdf: 197 entries
[01_5] uvm_users_guide_1.2.pdf: wrote 2178 records to c:\Users\41v1r\NEU\NLP\UVM-RAG\work\work_manual\json_out\uvm_blocks_augmented.jsonl
[TOC] UVM_Class_Reference_Manual_1.2.pdf: 169 entries
[01_5] UVM_Class_Reference_Manual_1.2.pdf: wrote 12897 records to c:\Users\41v1r\NEU\NLP\UVM-RAG\work\work_class_reference\json_out\uvm_class_reference_blocks_augmented.jsonl
